# Quantum Hardware Verification

**Purpose:** This notebook strictly serves to verify the trained PyTorch quantum-classical hybrid model using **real IBM Quantum hardware** for pure inference. The evaluation on physical Quantum Processing Units (QPUs) is essential to validate the robustness of the Quantum Neural Network (QNN) under real-world noise models, compared to statevector simulation.

**Constraints & Expectations:**
- 🚫 **No Training:** Pure evaluation phase. No gradients, optimizers, or weights updates (`eval()` + `torch.no_grad()`).
- ⚡ **Real Hardware Execution:** Utilizes PennyLane's `qiskit.remote` device to compile workloads and submit them to IBM Cloud infrastructure.
- 🔐 **Configuration:** Requires Colab Secrets `IBM_QUANTUM_TOKEN` for IBM Qiskit Runtime service authentication.
- 📁 **Storage:** Mounts Google Drive to load pre-trained frozen classical weights and the optimized Parameterized Quantum Circuit (PQC) parameters.

In [1]:
# Colab setup: mount Drive, clone repo, checkout main branch
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/lburdman/qnn-transfer-learning.git'
REPO_PATH = Path('/content/qnn-transfer-learning')
BRANCH = 'main'

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not REPO_PATH.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_PATH)], check=True)
    os.chdir(REPO_PATH)
    subprocess.run(['git', 'fetch'], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', 'origin', BRANCH], check=True)
else:
    os.chdir(Path('.'))

sys.path.append(str(Path.cwd() / 'src'))
print('Working dir:', Path.cwd())
print('Python path updated with src/')


Mounted at /content/drive
Working dir: /content/qnn-transfer-learning
Python path updated with src/


In [2]:
!pip install -U qiskit qiskit-ibm-runtime pennylane pennylane-qiskit

import os
from pathlib import Path
import torch
from qiskit_ibm_runtime import QiskitRuntimeService

if IN_COLAB:
    from google.colab import userdata
    api_key = userdata.get("IBM_QUANTUM_TOKEN")
    instance_crn = userdata.get("IBM_CRN")
else:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv("IBM_QUANTUM_TOKEN")
    instance_crn = os.getenv("IBM_CRN")

if not api_key:
    raise ValueError(
        "❌ IBM_QUANTUM_TOKEN not found.\n"
        "Create a Secret with that name in Colab, or use a .env file locally."
    )
else:
    print("✅ IBM Quantum API key found.")

instance_str = "open-instance" if not instance_crn else instance_crn

# Strictly save account to avoid repeated authentication warnings and set default
QiskitRuntimeService.save_account(
    channel="ibm_quantum",
    token=api_key,
    instance=instance_str,
    set_as_default=True,
    overwrite=True
)

service = QiskitRuntimeService()
print(f"✅ Connected OK to Qiskit Runtime (Instance: {instance_str}).")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.9/249.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.7/378.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 935.6/935.6 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB

qiskit_runtime_service._discover_account:WARNING:2026-02-24 23:26:40,243: Loading account with the given token. A saved account will not be used.


✅ IBM Quantum API key encontrada.
✅ Conectado OK a Qiskit Runtime.


## 1. Environment & Architecture Configuration
We declare the model architecture parameters, testing dataset targets, and the quantity of quantum measurements (`SHOTS`) desired per inference trace. The classes to evaluate are dynamically pulled from the model's experimental `config.json`.

In [3]:
import json
import os

# --- CONFIGURE THESE PATHS ---
MODEL_PATH = "/content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/model.pt"
TEST_DATA_DIR = "/content/drive/MyDrive/CREMAD/Embeddings/ResNet18/val"
BATCH_SIZE = 8
SHOTS = 1024

# Read config from model folder to filter data
config_path = os.path.join(os.path.dirname(MODEL_PATH), "config.json")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        config_data = json.load(f)

    if "selected_classes" in config_data:
        selected_classes = config_data["selected_classes"]
    elif "config" in config_data and "selected_classes" in config_data["config"]:
        selected_classes = config_data["config"]["selected_classes"]
    else:
        selected_classes = ['ANG', 'SAD']
else:
    selected_classes = ['ANG', 'SAD']

print(f"Loaded config from {config_path}")
print(f"Filtering dataset for classes: {selected_classes}")

from src.model_builder import build_model
from src.dataset import AudioFeatureDataset
from torch.utils.data import DataLoader
from torchvision import transforms


Loaded config from /content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/config.json
Filtering dataset for classes: ['ANG', 'SAD']


## 2. IBM Quantum Runtime Service Connectivity
Here we establish an active network session with the IBM Quantum infrastructure. By querying the `QiskitRuntimeService`, we filter out pure classical simulators and specifically request physical backend availability. The `qiskit.remote` PennyLane interface then connects the PyTorch QNode directly to the selected QPU (e.g. `ibm_fez` or `ibm_kyoto`).

In [8]:
from qiskit_ibm_runtime import QiskitRuntimeService
import pennylane as qml

# Authenticate explicitly using the defined instance and proper platform channel
instance_str = "open-instance" if not instance_crn else instance_crn

try:
    service = QiskitRuntimeService(token=api_key, channel="ibm_quantum_platform", instance=instance_str)
except Exception as e:
    QiskitRuntimeService.save_account(token=api_key, channel="ibm_quantum_platform", instance=instance_str, overwrite=True)
    service = QiskitRuntimeService(channel="ibm_quantum_platform", instance=instance_str)

# strict QPU filtering
all_backends = service.backends(simulator=False, operational=True)
real_backends = []
for b in all_backends:
    # HARD REQUIREMENT: No simulators, must be operational
    if not b.configuration().simulator:
        real_backends.append(b)

if not real_backends:
    raise RuntimeError("❌ No operational real hardware backends available right now.")

# Sort by least pending jobs to optimize queue time
real_backends.sort(key=lambda b: b.status().pending_jobs)

print("Available Operational QPUs (sorted by queue):")
for b in real_backends:
    print(f" - {b.name} (Queue: {b.status().pending_jobs} jobs)")

# Select the most optimal backend object directly
selected_backend_object = real_backends[0]
BACKEND_NAME = selected_backend_object.name 

print(f"\n✅ Selected Optimal Backend: {BACKEND_NAME}")


qiskit_runtime_service.__init__:WARNING:2026-02-24 23:30:24,688: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-24 23:30:24,689: Loading instance: open-instance, plan: open


Available Real IBM Quantum Backends:
 - ibm_fez
 - ibm_torino
 - ibm_marrakesh
Selected Backend: ibm_fez


/usr/local/lib/python3.12/dist-packages/pennylane/devices/device_api.py:201: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


✅ PennyLane device connected to ibm_fez


> *Note:* By explicitly defining `instance="open-instance"` via `QiskitRuntimeService()`, we ensure that executing quantum circuits correctly attributes to your main workspace allocation. This successfully mitigates any IBM Console tracking bugs where jobs might visually disappear or fail under an unresolved ambiguous trial plan state.

### Native Runtime Smoke Test (Job Mode Only)

> **Important Open Plan Constraint:** We strictly use standalone **Job Mode** directly targeting the backend (`SamplerV2(backend=...)`), as Open Plan users are denied authorization to use exclusive `Session(...)` contexts.
> Additionally, we pass the raw `QuantumCircuit` *without* executing local `transpile()` or preset pass managers here. Recent `qiskit` versions contain a `stevedore` ProviderV1 fallback crash on local manual compilation. `SamplerV2` handles its own native backend instruction set translation remotely!

Submitting this standalone 1-qubit `qiskit` job yields a transparent string you can use to track the physical footprint inside the IBM Cloud UI.


In [ ]:
import inspect
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2

# Minimal 1-qubit circuit (Hadamard + Measure)
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

print(f"Compiling native smoke test for {BACKEND_NAME} using V2 PassManager...")

# Qiskit V2: generate ISA circuit for the selected backend target
pm = generate_preset_pass_manager(
    target=selected_backend_object.target,
    optimization_level=3
)
isa_qc = pm.run(qc)

print(f"Original circuit depth: {qc.depth()} | ISA circuit depth: {isa_qc.depth()}")
print(f"Original ops: {qc.count_ops()} | ISA ops: {isa_qc.count_ops()}")

print(f"Executing native smoke test directly via Job Mode for {BACKEND_NAME}...")

# ---- SamplerV2 init: version-safe (mode vs backend) ----
sig = inspect.signature(SamplerV2.__init__)
kwargs = {}

if "mode" in sig.parameters:
    kwargs["mode"] = selected_backend_object
elif "backend" in sig.parameters:
    kwargs["backend"] = selected_backend_object
else:
    raise TypeError(
        "SamplerV2.__init__ signature does not accept 'mode' or 'backend'. "
        "Check qiskit-ibm-runtime version and update the notebook accordingly."
    )

sampler = SamplerV2(**kwargs)

try:
    print("Submitting direct Sampler job to IBM Open Plan queue...")
    job = sampler.run([isa_qc], shots=SHOTS)
    job_id = job.job_id()

    print(f"\n🔥 NATIVE IBM RUNTIME JOB ID: {job_id}")
    print(f"➡️ Copy this Job ID to verify execution ran on {BACKEND_NAME}")

    try:
        print(f"Current Status: {job.status()}")
    except Exception:
        pass

    print(
        "\n(Note: We do not block on `.result()` here to save notebook execution time. "
        "The job will process in the IBM queue.)"
    )

except Exception as e:
    print(f"❌ Failed to submit Job-Mode Payload: {e}")


### How to verify in IBM Quantum Platform
1. Under your IBM Cloud/Quantum Platform dashboard, navigate to **Workloads** or **Jobs**.
2. Paste the `NATIVE IBM RUNTIME JOB ID` printed above into the search bar.
3. You will visually verify the target system is indeed the physical `ibm_` QPU and that the computation used standard physical quantum seconds.

> *Note:* Remaining free-tier or open plan quota time is not cleanly exposed via the public Python REST API in real-time. Please monitor your overarching limits in the IBM dashboard UI over the lifecycle of your executions.

---


### Initializing PyTorch QNode Binding
Now we bind PennyLane `qml.device` rigidly to the identical hardware object we just verified.

In [ ]:
# We now inject the strictly validated 'selected_backend_object' directly
# into the PyTorch `build_model` signature to natively bind the QNodes instead of manually defining `dev` here.
print(f"Preparing PyTorch QNode injection for strict backend: {BACKEND_NAME}")


## 3. Hybrid Model Instantiation & Topology Prep
The PyTorch environment reconstructs the Quantum-Classical transfer learning architecture. It builds the frozen classical extractor (ResNet18 embeddings) leading into the dense quantum layer (our defined Variational Quantum Circuit). The saved experimental weights are directly mapped into the state. 

In [15]:
# --- GPU ENFORCEMENT ---
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ CUDA is available. Enforcing GPU for classical backbone.")
else:
    device = torch.device("cpu")
    print("⚠️ CUDA not available. Falling back to CPU.")

# Reconstruct class_names based on sorted selected_classes
class_names = sorted(selected_classes)

n_qubits = 2 # Ensure n_qubits is declared before use in model_config
# Using standard repo build_model logic
model_config = {
    'base_model': 'emb_resnet18',
    'quantum': True,
    'n_qubits': n_qubits,
    'q_depth': 3,
    'max_layers': 10,
    'q_delta': 0.01,
}

# Override using what we found in config if available
# The config_data dictionary is flat, so iterate directly over its items
if 'config_data' in locals():
    for k, v in config_data.items():
        # Only update/add keys that are relevant for the model_config
        if k in model_config or k == 'classical_model':
            model_config[k] = v

print("Instantiating Model...")

# --- FIX: Provide a dummy dataloader for model instantiation ---
# The build_model function needs to infer input_dim from dataloaders['train']
# For emb_resnet18, the embedding dimension is 512.
from torch.utils.data import Dataset, DataLoader
class DummyDataset(Dataset):
    def __len__(self):
        return 1
    def __getitem__(self, idx):
        # Return a dummy tensor with the expected embedding dimension (512 for ResNet18)
        return torch.randn(512), 0 # Dummy input and label

dummy_dataloader = DataLoader(DummyDataset(), batch_size=1) # batch_size doesn't matter for shape inference
dataloaders_for_build = {"train": dummy_dataloader}

model = build_model(
    model_config, 
    class_names, 
    dataloaders=dataloaders_for_build, 
    device=device,
    quantum_device="qiskit.remote",
    quantum_device_kwargs={"backend": selected_backend_object, "shots": SHOTS}
)

assert "default.qubit" not in str(model), "❌ QNode failed to bind physical hardware."
print("✅ Fully connected quantum hardware bound to PyTorch model.")

# LOAD WEIGHTS
if not os.path.exists(MODEL_PATH):
    print(f"❌ Could not find model at {MODEL_PATH}")
else:
    print(f"Loading weights from {MODEL_PATH}...")
    state_dict = torch.load(MODEL_PATH, map_location=device)
    if isinstance(state_dict, torch.nn.Module):
        model = state_dict
    else:
        model.load_state_dict(state_dict)

    model.to(device)
    model.eval() # CRITICAL for inference!
    print("✅ Model loaded and set to evaluation mode.")
    
    # --- STRICT MODEL WEIGHT INSTRUMENTATION ---
    print("\n--- Model Weight Validation ---")
    is_module = isinstance(state_dict, torch.nn.Module)
    print(f"Loaded object type: {'Module' if is_module else 'state_dict'}")
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters: {total_params:,} | Trainable: {trainable_params:,}")
    
    print("\nQuantum Head Parameters:")
    q_weights_found = False
    for name, param in model.named_parameters():
        if "qnode" in name.lower() or "quantum" in name.lower() or param.shape == (model_config['q_depth'], model_config['n_qubits']):
            q_weights_found = True
            print(f" - Found Quantum Weight Tensor: '{name}'")
            print(f"   Shape: {list(param.shape)}")
            print(f"   Stats: min={param.min().item():.4f}, max={param.max().item():.4f}, mean={param.mean().item():.4f}, std={param.std().item():.4f}")
            print(f"   First 10 values (flattened): {param.flatten()[:10].tolist()}")
    
    if not q_weights_found:
        print("⚠️ Explicit quantum parameter names not found. Dumping top-level state_dict keys:")
        print(list(model.state_dict().keys())[:15])


# DATA PREPARATION (Filtering subset by reading from subfolders)
try:
    filepaths = []
    labels = []
    for i, cls_name in enumerate(class_names):
        cls_dir = os.path.join(TEST_DATA_DIR, cls_name)
        if os.path.exists(cls_dir):
            # Tomamos todos los archivos del directorio de la clase filtrada
            for file in os.listdir(cls_dir):
                filepaths.append(os.path.join(cls_dir, file))
                labels.append(i)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])

    test_dataset = AudioFeatureDataset(filepaths, labels, base_model=model_config['base_model'])
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True)
    dataloaders = {'test': test_loader}
    print(f"✅ Data loaded: {len(test_dataset)} valid samples found.")
except Exception as e:
    print(f"⚠️ Warning: Data load failed. Make sure {TEST_DATA_DIR} exists with valid samples.")
    print(e)
    test_loader = []
    dataloaders = {'test': test_loader}


Instantiating Model...
Loading weights from /content/drive/MyDrive/CREMAD/Models/emb_resnet18/emb_resnet18_quantum/emb_resnet18_quantum_bi_0004/model.pt...
✅ Model loaded and set to evaluation mode.
✅ Data loaded: 481 valid samples found.


## 4. Hardware Inference Execution
This sends our filtered dataset to IBM Quantum Hardware.

### Inference Probe (Hardware Profiling)
Because PennyLane inherently obscures the underlying Qiskit `RuntimeJob` objects internally, we will manually intercept the exact PyTorch-generated parameters (weights and embeddings) for the very first sample. We will bind them to a pure Qiskit `QuantumCircuit`, transpile it for the ISA, and shoot it directly via `SamplerV2`. This acts as an "Inference Probe" giving you a highly visible, trackable `job_id` mirroring the exact ML workload.

In [ ]:
import time
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector

print("Grabbing 1 sample to act as the hardware compilation probe...")
sample_inputs, sample_labels = next(iter(dataloaders['test']))
sample_inputs = sample_inputs.to(device)

# 1. Profile Classical Extraction
t0 = time.time()
with torch.no_grad():
    if hasattr(model, 'forward_features'):
        embeddings = model.forward_features(sample_inputs[0:1])
    elif isinstance(model, torch.nn.Sequential):
        embeddings = model[:-1](sample_inputs[0:1])
    else:
        embeddings = sample_inputs[0:1]
t1 = time.time()
print(f"\n⏱️ Classical Extractor Time: {t1-t0:.4f}s")
print(f"Embedding bottleneck size sent to QPU: {embeddings.shape}")

# 2. Extract Quantum Weights for Binding
print("\nLocating QNode Weights for Native Binding...")
q_weight_tensor = None
for name, param in model.named_parameters():
    if "qnode" in name.lower() or "quantum" in name.lower() or getattr(param, 'shape', None) == (model_config['q_depth'], model_config['n_qubits']):
        q_weight_tensor = param.detach().cpu()
        break
if q_weight_tensor is None:
    raise RuntimeError("Could not locate the trained quantum weights in the model state dict.")

# 3. Construct 1:1 PennyLane Mirrored Circuit (AngleEmbedding + BasicEntanglerLayers)
probe_qc = QuantumCircuit(n_qubits)

# Parameter Vectors for dynamic bounding
x_params = ParameterVector('x', n_qubits)
w_params = ParameterVector('w', model_config['q_depth'] * n_qubits)

# AngleEmbedding (RX gates)
for i in range(n_qubits):
    probe_qc.rx(x_params[i], i)

# BasicEntanglerLayers
param_idx = 0
for layer in range(model_config['q_depth']):
    # Entangling ring
    for i in range(n_qubits):
        probe_qc.cx(i, (i + 1) % n_qubits)  # PennyLane default uses CNOTs in a ring
    # Rotations
    for i in range(n_qubits):
        probe_qc.rx(w_params[param_idx], i)
        param_idx += 1

probe_qc.measure_all()

# 4. Transpile
t2 = time.time()
pm_inf = generate_preset_pass_manager(target=selected_backend_object.target, optimization_level=3)
isa_probe = pm_inf.run(probe_qc)
t3 = time.time()
print(f"⏱️ ISA Compilation Time (Probe): {t3-t2:.4f}s")
print(f"Topology Match -> Depth Before: {probe_qc.depth()} | Depth After (ISA): {isa_probe.depth()}")

# 5. Bind Parameters
# Flatten bindings: x_params get the embeddings, w_params get the trained weights
emb_flat = embeddings[0].cpu().numpy()[:n_qubits] # Truncated to n_qubits if AngleEmbedding strictly maps 1-to-1
w_flat = q_weight_tensor.flatten().numpy()

# Ensure matching shapes
if len(emb_flat) < n_qubits:
    emb_flat = list(emb_flat) + [0.0] * (n_qubits - len(emb_flat)) # Pad if necessary

param_values = list(emb_flat) + list(w_flat)
print(f"Circuit Parameters: {probe_qc.num_parameters} | Bound Values Provided: {len(param_values)}")
assert probe_qc.num_parameters == len(param_values), "Parameter count mismatch!"

# 6. Submit the structural payload footprint directly
try:
    print("\nSubmitting Inference-Scaled Probe to IBM Open Plan Queue...")
    # SamplerV2 Requires a tuple of (circuit, parameter_values) if parameterized
    job_inf = sampler.run([(isa_probe, param_values)], shots=SHOTS)
    print(f"\n🔥 INFERENCE PROBE NATIVE JOB ID: {job_inf.job_id()}")
    print("➡️ You can now track the exact structural ML topology in your IBM dashboard.")
except Exception as e:
    print(f"❌ Probe payload failed: {e}")


In [16]:
# --- PROFILING INFERENCE TRACE ---
# We restrict the test batch size here because IBM Open Plan physical inference queueing can take many minutes per execution.

model.eval()

# Helper for timing and executing a batch
def execute_batch(loader_iter, num_batches, description):
    running_corrects = 0
    total_samples = 0
    total_qpu_time = 0.0
    
    print(f"\n🚀 {description} on REAL QPU: {BACKEND_NAME}...")
    
    with torch.no_grad():
        for i in range(num_batches):
            try:
                inputs, labels = next(loader_iter)
            except StopIteration:
                break
                
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            t_start = time.time()
            # 1. Classical extraction (Done implicitly by Sequential but we can time the full forward pass)
            try:
                outputs = model(inputs)
            except Exception as e:
                print(f"\n❌ Remote Physical Job Failed bounds during batch {i}: {e}")
                break
                
            elapsed = time.time() - t_start
            total_qpu_time += elapsed
            
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)
            total_samples += inputs.size(0)
            
            print(f"Batch {i+1} | Latency: {elapsed:.2f}s | Preds: {preds.tolist()} | True: {labels.tolist()}")
            
    if total_samples > 0:
        acc = running_corrects.double() / total_samples
        print(f"\n✅ {description} Complete! Evaluated {total_samples} samples.")
        print(f"📈 Batch Accuracy: {acc:.4f} | ⏱️ Total Time: {total_qpu_time:.2f}s")
    return total_samples

# Phase 1: Test 1 sample isolated
print("--- STAGE 1: Isolated 1-Sample Test ---")
from torch.utils.data import DataLoader, Subset
single_sample_loader = DataLoader(Subset(test_dataset, [0]), batch_size=1)
single_iter = iter(single_sample_loader)
execute_batch(single_iter, num_batches=1, description="Single Sample Run")

# Phase 2: Test larger batch
print("\n--- STAGE 2: Batched Evaluation ---")
# Reset standard loader iter
standard_iter = iter(test_loader) # Batch size is already 8
# Evaluate exactly 1 batch of size 8
execute_batch(standard_iter, num_batches=1, description="Batch=8 Run")


Executing batch of 8 samples on ibm_fez...

--- Inference Results ---
Sample 1:
  Predicted Class: SAD
  Confidence:      0.7875
  Actual Class:    SAD
Sample 2:
  Predicted Class: SAD
  Confidence:      0.9115
  Actual Class:    SAD
Sample 3:
  Predicted Class: ANG
  Confidence:      0.8806
  Actual Class:    ANG
Sample 4:
  Predicted Class: SAD
  Confidence:      0.9360
  Actual Class:    SAD
Sample 5:
  Predicted Class: ANG
  Confidence:      0.9231
  Actual Class:    ANG
Sample 6:
  Predicted Class: SAD
  Confidence:      0.9360
  Actual Class:    SAD
Sample 7:
  Predicted Class: SAD
  Confidence:      0.9265
  Actual Class:    SAD
Sample 8:
  Predicted Class: ANG
  Confidence:      0.9480
  Actual Class:    ANG

✅ Total Hardware Execution Time: 5.76 seconds


## 5. Global Validation Hardware Accuracy
The final objective is to feed the full validation subset collected from configuration (e.g., all samples within `ANG` and `SAD` splits) into the real Quantum Hardware pipeline. By accumulating the `argmax` classification traces across the entire loader, we calculate the absolute empirical accuracy of our model under real physical quantum noise characteristics.

In [ ]:
import time
import torch.nn.functional as F

print(f"Executing global evaluation over {len(test_dataset)} samples on {BACKEND_NAME}...")
print(f"🚀 Running purely physical inference on REAL QPU: {BACKEND_NAME}...")\nstart_time = time.time()

correct = 0
total = 0

with torch.no_grad():
    for batch_idx, (inputs, labels) in enumerate(test_loader):
        inputs = inputs.to(device)
        logits = model(inputs)
        
        probs = F.softmax(logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        
        batch_correct = (preds.cpu() == labels.cpu()).sum().item()
        correct += batch_correct
        total += labels.size(0)
        
        print(f"Batch {batch_idx+1}/{len(test_loader)} Processed | Batch Accuracy: {batch_correct/labels.size(0)*100:.1f}%")

end_time = time.time()
accuracy = 100 * correct / (total if total > 0 else 1)
print(f"\n✅ Global Hardware Execution Time: {end_time - start_time:.2f} seconds")
print(f"🎯 Global Hardware Inference Accuracy: {accuracy:.2f}% ({correct}/{total})")
